 # 📍 Address Extraction for Geocoding (2024 Voterfile)

This Python script retrieves the most recent 2024 voterfile snapshot, selecting only the address-related columns (street, city, state, ZIP).
The goal is to prepare these records for geocoding via a self-hosted Nominatim instance running on a local server.
The latitude and longitude results will later be used for spatial analysis on Florida’s voterfile data.

# Step 1: Retrieve Voter Address from Voterfile Database

This step queries the voterfile database to extract key address fields (street, city, state, ZIP) for all registered voters.  
The output will be used in subsequent geocoding steps to attach geographic coordinates for spatial analysis.

In [ ]:
import psycopg2

# Database connection parameters 
db_name = "fl_election"
db_user = "postgres"
db_password = "1434"
db_host = "192.168.86.36" # Change to server's IP when acessing remotely
db_port = "5432"

def connect_to_db():
    """ Establish a connection to PostgreSQL and return the connection & cursor. """
    
    try:
        print("Attempting to connect...")
        conn = psycopg2.connect(
            dbname = db_name,
            user = db_user,
            password = db_password,
            host = db_host,
            port = db_port
        )
        cur = conn.cursor()
        print("Connected to PostgreSQL")
        return conn, cur
    except Exception as e:
        print(f"Database connection error: {e}")
        return None, None
    
# Call the function to test connection
conn, cur = connect_to_db()

Attempting to connect...
Connected to PostgreSQL


In [ ]:
import pandas as pd

query = "select residence_address_1 street, residence_city_usps city, residence_zipcode zip_code, 'FL' AS state from voterfile.election_detail_2024"

df = pd.read_sql(query, conn)

/var/folders/h3/qc0834sx6gjb7174_439hbkc0000gn/T/ipykernel_78566/2907065733.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


# Step 2: Create Normalized `formatted_address` Column

This step constructs a unified `formatted_address` field from raw voterfile columns (street, city, state, and zip).  
Normalization includes:
- Stripping extra whitespace and applying consistent casing
- Converting full directional words (e.g., "Southwest") to abbreviations ("SW")
- Standardizing common street suffixes (e.g., "Avenue" → "Ave", "Street" → "St")

This ensures clean, consistent input for geocoding via Nominatim.

In [40]:
import re 
import pandas as pd

def normalize_address(row):

    # lowercase + strip white space
    street = str(row['street']).strip().lower()
    street = re.sub(r'\s+', ' ', street)  # normalize extra spaces
    city = str(row['city']).strip().title()
    state = str(row['state']).strip().upper()
    zip_code = str(row['zip_code']).strip().upper().zfill(5)
    
    directionals = {
        'northwest' : 'nw',
        'northwest' : 'ne',
        'southwest' : 'sw',
        'southeast' : 'se',
        'north' : 'n',
        'south' : 's',
        'east' : 'e',
        'west' : 'w',
    }

    for full, abbr in directionals.items():
        street = re.sub(r'\b' + full + r'\b', abbr, street)

    # Normaization street suffixes 
    suffixes = {
        'street' : 'st',
        'avenue' : 'ave',
        'boulevard' : 'blvd', 
        'road' : 'rd', 
        'drive' : 'dr', 
        'lane' : 'ln', 
        'court' : 'ct'
     }
    for full, abbr in suffixes.items():
        street = re.sub(r'\b' + full + r'\b', abbr, street)
    
    # combined the final address string
    full_address = f"{street.title()}, {city}, {state} {zip_code}"
    return full_address
    

df['formatted_address'] = df.apply(normalize_address, axis=1)


# Step 3: Create SQLite Database for Caching Geocoding Results

To handle the large size of the voterfile (~8 million rows), we use a local SQLite database to **cache geocoding results**.  
This prevents re-querying the same address multiple times and safeguards progress in case of interruptions (e.g., network errors, power loss).

The function below allows flexible creation of a `.db` file and table by specifying the table name and database name.

This cache will:
- Save lat/lon coordinates for previously queried addresses
- Track geocoding status (`success`, `no_match`, or `error`)
- Speed up repeated runs and make the process fault-tolerant

In [284]:
import sqlite3

def initialize_cache_db(table_name: str, db_name: str):
    conn = sqlite3.connect(db_name)
    cur = conn.cursor()

    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            address TEXT PRIMARY KEY,
            lat REAL,
            lon REAL,
            status TEXT
        )
    """)

    conn.commit()
    conn.close()
    print(f"✅ Table '{table_name}' created or already exists in database '{db_name}'!")

# Example usage:
# initialize_cache_db("geocode_cache", "geocache_test.db")
# initialize_cache_db("geocode_cache_rerun", "geocache_rerun.db")
initialize_cache_db("geocode_cache_rerun_threads", "geocache_rerun_threads.db")

✅ Table 'geocode_cache_rerun_threads' created or already exists in database 'geocache_rerun_threads.db'!


# Step 4: Geocode Addresses with Local Nominatim & Cache Results

This step defines a Python function that takes a single formatted address, sends a request to your locally hosted Nominatim instance, and caches the result in a custom SQLite database and table.

Parameters:
	•	address (str): A formatted address string, e.g., "1222 SW 103rd Ln, Miami, FL 33196"
	•	db_name (str): Name of the SQLite database file (e.g., "geocache_test.db")
	•	table_name (str): Name of the table where results will be cached (e.g., "geocode_cache")

In [75]:
import requests
import sqlite3

def geocode_address_cached(address, db_name, table_name, base_url="http://192.168.86.36:8080"):
    """Geocode a single address using local Nominatim instance and cache results in a custom DB/table."""
    if not address or pd.isna(address) or address.strip() == "":
        return None, None, "invalid_input"
    conn = sqlite3.connect(db_name)
    cur = conn.cursor()

    # Check if address is already in the cache
    cur.execute(f"SELECT lat, lon, status FROM {table_name} WHERE address = ?", (address,))
    row = cur.fetchone()

    if row:
        conn.close()
        return row[0], row[1], row[2]

    # Try geocoding via Nominatim
    try:
        response = requests.get(
            f"{base_url}/search",
            params={"q": address, "format": "json", "limit": 1},
            timeout=5
        )
        response.raise_for_status()
        data = response.json()

        if data:
            lat = float(data[0]["lat"])
            lon = float(data[0]["lon"])
            status = "success"
        else:
            lat, lon = None, None
            status = "no_match"

    except Exception as e:
        print(f"Error: {e} for {address}")
        lat, lon = None, None
        status = "error"

    # Cache the result regardless of outcome
    cur.execute(
        f"INSERT INTO {table_name} (address, lat, lon, status) VALUES (?, ?, ?, ?)",
        (address, lat, lon, status)
    )
    conn.commit()
    conn.close()

    return lat, lon, status

# Step 5: Apply Geocoding Function to Entire DataFrame

This method ensures you’re only querying uncached addresses and logging every result for future use.

📌 Warning: 8 million rows will take multiple days. Parallelism is number 1 prioritiy for v2 

What This Does:
	•	Progress tracking: tqdm.pandas() adds a real-time progress bar so you can monitor the status of the geocoding process.
	•	Row-wise geocoding: Each row’s formatted_address is passed to the geocoding function.
	•	Caching results: Lat, lon, and status (e.g. success, no_match, or error) are returned and stored in the DataFrame.
	•	Fault tolerance: If the script stops or fails, previously geocoded rows are already saved in your .db file and won’t be queried again.


In [ ]:
from tqdm import tqdm

tqdm.pandas()  # Enables progress bar for pandas apply

# Define your database and table names
db_name = "geocache_test.db"
table_name = "geocode_cache"

# Use lambda to pass additional arguments into the geocode function
df['lat'], df['lon'], df['status'] = zip(*df['formatted_address'].progress_apply(
    lambda addr: geocode_address_cached(addr, db_name, table_name)
))

# Step 6: Load Cached Geocode Results into a DataFrame

Now that we’ve cached millions of address lookups into an SQLite database, we want a fast and reusable way to reload those results for inspection or reuse without rerunning the full geocoding process.

In [53]:
import sqlite3
import pandas as pd

def load_table_as_df(db_name, table_name):
    """
    Load a table from a SQLite database into a pandas DataFrame.
    
    Args:
        db_name (str): The path to the SQLite database file.
        table_name (str): The name of the table to load.
    
    Returns:
        pd.DataFrame: The loaded DataFrame.
    """
    conn = sqlite3.connect(db_name)
    df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()
    
    print(f"Preview of '{table_name}' from '{db_name}':")
    print(df.head())
    
    return df

In [ ]:
# Initalize first db 
df_cache_1 = load_table_as_df("geocache_test.db", "geocode_cache")

Preview of 'geocode_cache' from 'geocache_test.db':
                                   address        lat        lon   status
0  2227 Nw 50Th Way, Gainesville, FL 32605  29.727271 -82.395669  success
1     24214 Nw 52Nd Ter, Alachua, FL 32615  29.667915 -82.398139  success
2   1813 Sw 65Th Dr, Gainesville, FL 32607  29.633976 -82.413143  success
3  2930 Sw 23Rd Ter, Gainesville, FL 32608  29.631097 -82.361451  success
4    8825 Nw 4Th Pl, Gainesville, FL 32607  29.656597 -82.401556  success


# Step 7: Filter and Save Missing Results for Reprocessing

After the initial geocoding pass, some addresses may still be missing latitude and longitude due to formatting inconsistencies or unmatched records. In this step, we isolate those records for further inspection and potentially retry geocoding with alternate formatting strategies.

In [ ]:
# Save separate df for address that did not get matching 
df_missing = df_cache_1[df_cache_1['status'] == 'no_match']
df_missing[['street', 'city', 'state_zip']] = df_missing['address'].str.extract(r'^(.*?),\s*(.*?),\s*([A-Z]{2}.*)$')

# Now split the state and zip_code
df_missing[['state', 'zip_code']] = df_missing['state_zip'].str.extract(r'([A-Z]{2})\s*(\d{5})')

# Drop the helper column
df_missing = df_missing.drop(columns=['state_zip'])


# Step 8: Apply Basic Address Formatting to Test Alternative Matching

After the initial geocoding run, some addresses may still fail due to over-normalization or inconsistent formatting.
We now generate a simpler, less aggressive version of the address string to test if more matches can be found.

In [ ]:
import re
def basic_normalize_address(row):
    # Clean and format each part
    street = str(row['street']).strip().lower()
    street = re.sub(r'\s+', ' ', street)  # normalize multiple spaces

    city = str(row['city']).strip().title()
    state = str(row['state']).strip().upper()
    zip_code = str(row['zip_code']).strip().zfill(5)

    # Combine into one formatted string
    full_address = f"{street.title()}, {city}, {state} {zip_code}"
    return full_address

df_missing['basic_formatted_address'] = df_missing.apply(basic_normalize_address, axis=1)



In [104]:
df_sample = df_missing.sample(n=100000, random_state=10)

In [ ]:
from tqdm import tqdm

tqdm.pandas()  # Enables progress bar for pandas apply

# Define your database and table names
db_name = "geocache_rerun.db"
table_name = "geocode_cache_rerun"

# Use lambda to pass additional arguments into the geocode function
df_sample['lat'], df_sample['lon'], df_sample['status'] = zip(*df_sample['basic_formatted_address'].progress_apply(
    lambda addr: geocode_address_cached(addr, db_name, table_name)
))

# Step 9: Filter and Save Missing Results for Reprocessing with Only Street Number

In this step, we focus on addresses that were not successfully geocoded and prepare a simplified version of the address using only the street number. This helps us test if minimal input (like house number) improves matching accuracy for problematic cases.

In [ ]:
tqdm.pandas()  # Enables progress bar for pandas apply

# Define your database and table names
db_name = "geocache_rerun.db"
table_name = "geocode_cache_rerun_street"

# Use lambda to pass additional arguments into the geocode function
df_sample['lat'], df_sample['lon'], df_sample['status'] = zip(*df_sample['street'].progress_apply(
    lambda addr: geocode_address_cached(addr, db_name, table_name)
))

In [89]:
# Initalize first db 
# initialize_cache_db("geocode_cache_rerun_street", "geocache_test.db")
df_cache_2 = load_table_as_df("geocache_rerun.db", "geocode_cache_rerun_street")

Preview of 'geocode_cache_rerun_street' from 'geocache_rerun.db':
                 address        lat        lon   status
0  3664   Sugarcreek Dr   27.974559 -82.369301  success
1   1690   Village Pkwy   27.266196 -80.430577  success
2      2405  NE 8th Ter   26.297307 -80.112356  success
3     8306   Dahlia Ave   27.896689 -82.364100  success
4      9600  NW 42nd St   26.177176 -80.211369  success


In [97]:
import requests
import sqlite3
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def threaded_geocode_worker(address, db_name, table_name, base_url="http://192.168.86.36:8080"):
    """
    Geocode a single address using a local Nominatim instance with SQLite caching (thread-safe).
    Each thread creates its own DB connection.
    """
    if not address or pd.isna(address) or address.strip() == "":
        return address, None, None, "invalid_input"

    try:
        conn = sqlite3.connect(db_name)
        cur = conn.cursor()

        cur.execute(f"SELECT lat, lon, status FROM {table_name} WHERE address = ?", (address,))
        row = cur.fetchone()

        if row:
            conn.close()
            return address, row[0], row[1], row[2]

        response = requests.get(
            f"{base_url}/search",
            params={"q": address, "format": "json", "limit": 1},
            timeout=5
        )
        response.raise_for_status()
        data = response.json()

        if data:
            lat = float(data[0]["lat"])
            lon = float(data[0]["lon"])
            status = "success"
        else:
            lat, lon = None, None
            status = "no_match"

    except Exception as e:
        print(f"Error: {e} for {address}")
        lat, lon = None, None
        status = "error"
    finally:
        try:
            cur.execute(
                f"INSERT INTO {table_name} (address, lat, lon, status) VALUES (?, ?, ?, ?)",
                (address, lat, lon, status)
            )
            conn.commit()
            conn.close()
        except:
            pass  # Optional: handle errors writing to DB

    return address, lat, lon, status

def threaded_geocode_dataframe(df, address_col, db_name, table_name, max_workers=10):
    """
    Apply threaded geocoding to a DataFrame.
    Adds lat/lon/status columns directly.
    """
    tqdm.pandas(desc="Geocoding")
    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_address = {
            executor.submit(threaded_geocode_worker, addr, db_name, table_name): addr
            for addr in df[address_col]
        }

        for future in tqdm(as_completed(future_to_address), total=len(future_to_address)):
            result = future.result()
            results.append(result)

    # Unpack results directly into new columns
    lats = [r[1] for r in results]
    lons = [r[2] for r in results]
    statuses = [r[3] for r in results]

    df = df.copy()
    df["lat"] = lats
    df["lon"] = lons
    df["status"] = statuses

    return df


In [79]:
# Example input
db_name = "geocache_rerun_threads.db"
table_name = "geocode_cache_rerun_threads"
address_col = "street"  # or 'formatted_address'

In [ ]:
df_threaded = threaded_geocode_dataframe(
    df_sample,
    address_col=address_col,
    db_name=db_name,
    table_name=table_name,
    max_workers=10  # or however many threads your system can handle efficiently
)

In [99]:
df_threaded.head()

,address,lat,lon,status,street,city,state,zip_code,basic_formatted_address
2240596,"6618 Cleveland Rd, Jacksonville, FL 322091606",30.473236,-86.128903,success,6618 Cleveland Rd,Jacksonville,FL,32209,"6618 Cleveland Rd, Jacksonville, FL 32209"
6556452,"229 Yarmouth Rd, Fern Park, FL 327303011",27.646350,-80.478309,success,229 Yarmouth Rd,Fern Park,FL,32730,"229 Yarmouth Rd, Fern Park, FL 32730"
4525898,"9120 Lytham Ct, Orlando, FL 32819",30.400125,-87.302926,success,9120 Lytham Ct,Orlando,FL,32819,"9120 Lytham Ct, Orlando, FL 32819"
6865938,"700 Fourth Ave, Wildwood, FL 34785",26.572716,-81.876302,success,700 Fourth Ave,Wildwood,FL,34785,"700 Fourth Ave, Wildwood, FL 34785"
3225090,"4615 Dunnie Dr, Tampa, FL 33614",26.614954,-80.168125,success,4615 Dunnie Dr,Tampa,FL,33614,"4615 Dunnie Dr, Tampa, FL 33614"


# Step 10: Uploaded and processed geotagged addresses to fl_election database

In [ ]:
db1 = pd.read_sql("SELECT * FROM my_table", sqlite3.connect("geocache_test.db"))

# Read from second DB
db2 = pd.read_sql("SELECT * FROM my_table", sqlite3.connect("geocache_rerun.db"))

# Combine them--
combined_db = pd.concat([db1, db2], ignore_index=True)

# Testing Simple Testing API Calls 

In [47]:
import requests

address = "2758   BELLVIEW AVE "
url = f"http://192.168.86.36:8080/search"
params = {
    "q": address,
    "format": "json",
    "limit": 1
}

response = requests.get(url, params=params)
data = response.json()

if data:
    print("Latitude:", data[0]["lat"])
    print("Longitude:", data[0]["lon"])
else:
    print("No match found.")

Latitude: 27.882999
Longitude: -81.572588
